# 09 Stacking Ensemble Final

In [ ]:

import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")

import json
import tempfile
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, roc_curve
)
from mlflow.models import infer_signature
from scipy.stats import randint

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 26
TEST_SIZE = 0.2
EXPERIMENT_NAME = "Current Stress"


def resolve_repo_root():
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / "nostressia-machine-learning").exists() and (parent / "nostressia-backend").exists():
            return parent
    return cwd

REPO_ROOT = resolve_repo_root()
TRACKING_URI = "file:" + str((REPO_ROOT / "mlruns").resolve()).replace("\\", "/")
DATASET_PATH = REPO_ROOT / "nostressia-machine-learning" / "Current-Stress" / "datasets" / "raw" / "student_lifestyle_dataset.csv"
NOTEBOOK_PATH = Path.cwd() / "09_stacking_ensemble_final.ipynb"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


def categorize_academic_performance(gpa):
    if gpa >= 3.5:
        return "Excellent"
    elif 3.0 <= gpa < 3.5:
        return "Good"
    elif 2.0 <= gpa < 3.0:
        return "Fair"
    return "Poor"


def load_preprocess_dataset(dataset_path=DATASET_PATH):
    df = pd.read_csv(dataset_path)
    raw_df = df.copy()

    df["Academic_Performance"] = df["GPA"].apply(categorize_academic_performance)
    mapping_stress = {"Low": 0, "Moderate": 1, "High": 2}
    mapping_performance = {"Poor": 0, "Fair": 1, "Good": 2, "Excellent": 3}

    df["Stress_Level_Encoded"] = df["Stress_Level"].map(mapping_stress)
    df["Academic_Performance_Encoded"] = df["Academic_Performance"].map(mapping_performance)
    df = df.drop(columns=["Stress_Level", "Academic_Performance"])

    feature_cols = [c for c in df.columns if c not in ["Stress_Level_Encoded", "Student_ID"]]
    X = df[feature_cols].copy()
    y = df["Stress_Level_Encoded"].copy()
    return raw_df, df, X, y


def split_data(X, y):
    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)


def log_common_params(feature_set, imbalance_handling="no", tuning_strategy="none"):
    mlflow.log_params({
        "dataset_path": str(DATASET_PATH),
        "feature_set": feature_set,
        "preprocessing": "Academic_Performance derivation + ordinal encoding + Student_ID dropped",
        "scaler": "RobustScaler for LR/stacking where applicable",
        "imputation": "none",
        "split_test_size": TEST_SIZE,
        "split_random_state": RANDOM_STATE,
        "split_stratify": True,
        "imbalance_handling": imbalance_handling,
        "tuning_strategy": tuning_strategy,
        "tracking_uri": TRACKING_URI,
    })


def compute_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_accuracy": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba, multi_class="ovr")
    return metrics


def build_eval_artifacts(model, X_test, y_test, artifact_dir, prefix="eval"):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None

    metrics = compute_metrics(y_test, y_pred, y_proba)

    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="PuBuGn", colorbar=False)
    ax.set_title(f"{prefix} - Confusion Matrix")
    fig.tight_layout()
    fig.savefig(artifact_dir / f"{prefix}_confusion_matrix.png", dpi=150)
    plt.close(fig)

    if y_proba is not None:
        y_bin = label_binarize(y_test, classes=np.unique(y_test))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), y_proba.ravel())
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label="ROC micro-average")
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.set_title(f"{prefix} - ROC Curve")
        ax.legend()
        fig.tight_layout()
        fig.savefig(artifact_dir / f"{prefix}_roc_curve.png", dpi=150)
        plt.close(fig)

    (artifact_dir / f"{prefix}_classification_report.txt").write_text(
        classification_report(y_test, y_pred, digits=4), encoding="utf-8"
    )

    pd.DataFrame({"y_true": y_test.values, "y_pred": y_pred}).head(100).to_csv(
        artifact_dir / f"{prefix}_sample_predictions.csv", index=False
    )

    return metrics


In [2]:

raw_df, processed_df, X, y = load_preprocess_dataset()
X_train, X_test, y_train, y_test = split_data(X, y)
base_estimators = [
    ('lr', Pipeline(steps=[('scaler', RobustScaler()), ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))])),
    ('dt', DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=8, min_samples_split=4)),
    ('rf', RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=350, class_weight='balanced', n_jobs=-1)),
]
meta_estimator = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
stack_model = StackingClassifier(estimators=base_estimators, final_estimator=meta_estimator, cv=5, passthrough=False)

with mlflow.start_run(run_name='Stacking Ensemble - Final'):
    mlflow.set_tags({'module':'current-stress','feature_group':'all','final_model':'true'})
    log_common_params(feature_set='all', imbalance_handling='class_weight', tuning_strategy='manual_from_best_runs')
    mlflow.log_param('base_estimators', 'lr_scaled, dt_tuned, rf_tuned_like')
    mlflow.log_param('meta_estimator', 'logistic_regression')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(stack_model, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
    mlflow.log_metric('cv_f1_weighted_mean', float(np.mean(cv_scores)))
    mlflow.log_metric('cv_f1_weighted_std', float(np.std(cv_scores)))
    stack_model.fit(X_train, y_train)

    with tempfile.TemporaryDirectory() as td:
        art = Path(td)
        metrics = build_eval_artifacts(stack_model, X_test, y_test, art, prefix='test')
        mlflow.log_metrics(metrics)
        sig = infer_signature(X_train.head(20), stack_model.predict(X_train.head(20)))
        mlflow.sklearn.log_model(stack_model, 'model', signature=sig, input_example=X_train.head(5))
        joblib.dump({'pipeline': stack_model, 'feature_names': list(X.columns)}, art/'current_stress_stacking_final.joblib')
        mlflow.log_artifact(str(NOTEBOOK_PATH), artifact_path='code')
        mlflow.log_artifacts(str(art), artifact_path='artifacts')


2026/02/20 14:00:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
